# Application 2 — Democracy, growth, and formal treatment switching

Complete replication notebook for **pydrlpdid 0.7.2**.

Expected Windows directory:

`C:\Users\danie\OneDrive\1 - Pesquisas\0 - DRLPDID\pydrlpdid-0.7.2`

The notebook contains all application code and produces the complete set
of tables, diagnostics, and figures used by the former 0.6.4 notebook.
It deliberately separates three empirical objects:

1. **Published Figure 4 replication.** A literal replication of the four
   specifications in Dube et al.; this is a literature benchmark.
2. **Dube-compatible onset design (CCC1).** The published event/control
   eligibility rules are held fixed while the five semiparametric
   estimators are computed by the 0.7.2 numerical engine.
3. **Formal sustained-switch-in design.** The public 0.7.2 switching API
   imposes an observed clean history, treatment sustained through each
   horizon, and stable-zero plus stabilized stable-one controls.

The CCC1 and formal blocks estimate different causal objects. The
application-specific CCC1 bridge does not add a CCC1 option to the public
package API.


## 0. Run configuration


In [ ]:
from pathlib import Path

PROJECT_DIR_WINDOWS = Path(
    r"C:\Users\danie\OneDrive\1 - Pesquisas\0 - DRLPDID\pydrlpdid-0.7.2"
)

OUTPUT_FOLDER = "application_2_dube_v072"
EXPECTED_DATA_SHA256 = (
    "097899861dc11e11256105dee814883e57b818b1e6627c16dd8676c36192bbe8"
)

H_PRE_DUBE = 20
H_POST_DUBE = 30
HISTORY_LENGTH = 20
POLICY_WINDOW = (0, 10)
DISPLAY_ONLY_LEADS = {-4, -3, -2}
BASE_PERIOD = -1

FORMAL_H_PRE = 10
FORMAL_H_POST = 15
FORMAL_L = 20
FORMAL_L9 = 9

ALPHA = 0.05
Z_975 = 1.959963984540054
N_MULTIPLIER = 999
MULTIPLIER_SEED = 20260727

RUN_FIGURE4_REPLICATION = True
RUN_DUBE_COMPATIBLE = True
RUN_FORMAL_SWITCHING = True
RUN_FORMAL_L9 = True


## 1. Local environment, data resolution, and reproducibility gate


In [ ]:
import hashlib
import importlib.metadata as importlib_metadata
import json
import platform
import shutil
import sys
import time
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("default")


def locate_project_root(preferred: Path) -> Path:
    if (preferred / "src" / "pydrlpdid" / "__init__.py").exists():
        return preferred.resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "pydrlpdid" / "__init__.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find pydrlpdid-0.7.2. Run this notebook from the "
        "project root or update PROJECT_DIR_WINDOWS."
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


PROJECT_DIR = locate_project_root(PROJECT_DIR_WINDOWS)
SRC_DIR = PROJECT_DIR / "src"
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / OUTPUT_FOLDER
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pydrlpdid
from pydrlpdid import DRLPDID
from pydrlpdid.drlpdid import (
    _compute_dr_generic,
    _compute_dr_improved,
    _compute_ipw,
    _compute_ra,
)

if pydrlpdid.__version__ != "0.7.2":
    raise RuntimeError(
        "This notebook requires pydrlpdid 0.7.2, but loaded "
        f"{pydrlpdid.__version__} from {pydrlpdid.__file__}."
    )
PACKAGE_FILE = Path(pydrlpdid.__file__).resolve()
if PROJECT_DIR not in PACKAGE_FILE.parents:
    raise RuntimeError(
        "A different copy of pydrlpdid was imported. Restart the kernel "
        "and run this notebook from the local 0.7.2 project root."
    )

PACKAGE_SOURCE_HASHES = {
    filename: sha256_file(PACKAGE_FILE.parent / filename)
    for filename in [
        "__init__.py",
        "drlpdid.py",
        "_inference.py",
        "_panel_utils.py",
        "_results.py",
    ]
}

data_candidates = [
    DATA_DIR / "DDCGdata_final.dta",
    PROJECT_DIR / "audit_dube_data" / "data" / "DDCGdata_final.dta",
    PROJECT_DIR / "Aplicacao_2_Dube_DRLPDID_v12_4"
    / "audit_dube_data" / "data" / "DDCGdata_final.dta",
    PROJECT_DIR.parent / "pydrlpdid-0.6.4" / "data" / "DDCGdata_final.dta",
]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)

if DATA_PATH is None:
    zip_candidates = [
        PROJECT_DIR / "Aplicacao_2_Dube_DRLPDID_v12_4.zip",
        PROJECT_DIR / "upload" / "Aplicacao_2_Dube_DRLPDID_v12_4.zip",
        PROJECT_DIR.parent / "Aplicacao_2_Dube_DRLPDID_v12_4.zip",
    ]
    bundle_zip = next((path for path in zip_candidates if path.exists()), None)
    if bundle_zip is not None:
        with zipfile.ZipFile(bundle_zip) as archive:
            members = [
                name for name in archive.namelist()
                if name.endswith("/DDCGdata_final.dta")
                or name == "DDCGdata_final.dta"
            ]
            if len(members) != 1:
                raise RuntimeError(
                    "The data bundle must contain exactly one "
                    "DDCGdata_final.dta."
                )
            DATA_PATH = DATA_DIR / "DDCGdata_final.dta"
            with archive.open(members[0]) as source_file, DATA_PATH.open("wb") as target:
                shutil.copyfileobj(source_file, target)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Copy DDCGdata_final.dta to the 0.7.2 data directory, or place "
        "Aplicacao_2_Dube_DRLPDID_v12_4.zip in the project root."
    )
if sha256_file(DATA_PATH) != EXPECTED_DATA_SHA256:
    raise RuntimeError("DDCGdata_final.dta does not match the certified hash.")

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UnicodeWarning)
    raw = pd.read_stata(
        DATA_PATH,
        columns=["wbcode2", "year", "y", "dem"],
        convert_categoricals=False,
    )
raw = raw.sort_values(["wbcode2", "year"]).reset_index(drop=True)
if raw.duplicated(["wbcode2", "year"]).any():
    raise RuntimeError("Duplicate country-year rows were found.")
for lag in range(1, 5):
    raw[f"lag{lag}y"] = raw.groupby("wbcode2", sort=False)["y"].shift(lag)

data_audit = {
    "data_path": str(DATA_PATH.resolve()),
    "sha256": sha256_file(DATA_PATH),
    "rows": int(len(raw)),
    "countries": int(raw["wbcode2"].nunique()),
    "year_min": int(raw["year"].min()),
    "year_max": int(raw["year"].max()),
    "periods": int(raw["year"].nunique()),
    "missing_y": int(raw["y"].isna().sum()),
    "missing_dem": int(raw["dem"].isna().sum()),
}
expected_shape = {
    "rows": 9384,
    "countries": 184,
    "year_min": 1960,
    "year_max": 2010,
    "periods": 51,
}
for key, expected in expected_shape.items():
    if data_audit[key] != expected:
        raise RuntimeError(
            f"Data audit failed for {key}: "
            f"{data_audit[key]} != {expected}."
        )

(OUTPUT_DIR / "data_audit.json").write_text(
    json.dumps(data_audit, indent=2), encoding="utf-8"
)
print("Project:", PROJECT_DIR)
print("pydrlpdid:", pydrlpdid.__version__, "from", PACKAGE_FILE)
print("Data:", DATA_PATH)
print("Output:", OUTPUT_DIR)
print(json.dumps(data_audit, indent=2))


## 2. Empirical targets

### Published Figure 4 replication

The first block reproduces the four published specifications as a
literature benchmark. It preserves their original regressions and
reported variance conventions; it is not a call to the DRLPDID estimator.

### Dube-compatible onset target

An event at year \(t\) is an observed \(0\to1\) transition. A comparison
observation remains at zero between \(t-1\) and \(t\). Both event and
comparison observations have no observed transition during the preceding
\(L=20\) years. Pre-sample transition lags pass this requirement under the
available-history convention. Eligibility does not condition on treatment
after \(t\).

The long difference is

\[
\Delta_hY_{ct}=Y_{c,t+h}-Y_{c,t-1}.
\]

Four predetermined GDP lags and calendar-year indicators enter both
nuisance bases. The base period \(h=-1\) is zero by construction. With the
four outcome lags, \(h=-2,-3,-4\) are not separately identified; they are
displayed at zero only to match the published graph and are excluded from
estimation and the simultaneous-inference family.

### Formal sustained-switch-in target

The formal block requires a fully observed clean \(L\)-year history,
treatment sustained through \(t+h\), and all-stayer controls that are
stable at zero or stable at one after effect stabilization. This is the
public `DRLPDID(design="switching")` estimand in version 0.7.2, not the
CCC1 onset estimand.


## 3. Shared Figure 4 and Dube-compatible helpers


In [ ]:
def shift(df, column, periods):
    return df.groupby("wbcode2", sort=False)[column].shift(periods)


def design_matrix(d, include_treatment=True):
    cols = []
    if include_treatment:
        cols.append(d["tdemoc"].to_numpy(float)[:, None])
    cols.append(d[[f"lag{k}y" for k in range(1, 5)]].to_numpy(float))
    years = pd.get_dummies(d["year"], dtype=float).to_numpy()
    cols.append(years)
    return np.column_stack(cols)


def ols_inference(d):
    y = d["long_y"].to_numpy(float)
    x = design_matrix(d)
    beta = np.linalg.lstsq(x, y, rcond=None)[0]
    u = y - x @ beta
    bread = np.linalg.pinv(x.T @ x)
    n, k = x.shape
    hc_meat = (x * u[:, None]).T @ (x * u[:, None])
    hc = bread @ hc_meat @ bread * n / max(n - k, 1)
    meat = np.zeros((k, k))
    for _, idx in d.groupby("wbcode2").indices.items():
        score = x[idx].T @ u[idx]
        meat += np.outer(score, score)
    g = d["wbcode2"].nunique()
    # Article influence-function formula: no parameter-count degrees-of-freedom factor.
    correction = 1.0
    cluster = bread @ meat @ bread * correction
    return beta[0], np.sqrt(hc[0, 0]), np.sqrt(cluster[0, 0])


def ra_atet(d):
    controls = d["tdemoc"].eq(0).to_numpy()
    treated = ~controls
    y = d["long_y"].to_numpy(float)
    x = design_matrix(d, include_treatment=False)
    xc, yc = x[controls], y[controls]
    beta = np.linalg.lstsq(xc, yc, rcond=None)[0]
    theta = np.mean(y[treated] - x[treated] @ beta)

    # Joint M-estimator for beta and the treated residual mean.
    k = x.shape[1]
    a = np.zeros((k + 1, k + 1))
    a[:k, :k] = -(xc.T @ xc)
    a[k, :k] = -(x[treated].sum(axis=0))
    a[k, k] = -treated.sum()
    scores = {}
    for country, idx in d.groupby("wbcode2").indices.items():
        idx = np.asarray(idx)
        c = controls[idx]
        t = treated[idx]
        s = np.zeros(k + 1)
        if c.any():
            ic = idx[c]
            s[:k] = x[ic].T @ (y[ic] - x[ic] @ beta)
        if t.any():
            it = idx[t]
            s[k] = np.sum(y[it] - x[it] @ beta - theta)
        scores[country] = s
    ainv = np.linalg.pinv(a)
    cluster_contrib = np.array([(-ainv @ s)[-1] for s in scores.values()])
    g = len(scores)
    se_cluster = np.sqrt(np.sum(cluster_contrib**2) * g / (g - 1))

    row_contrib = []
    for j in range(len(d)):
        s = np.zeros(k + 1)
        if controls[j]:
            s[:k] = x[j] * (y[j] - x[j] @ beta)
        else:
            s[k] = y[j] - x[j] @ beta - theta
        row_contrib.append((-ainv @ s)[-1])
    row_contrib = np.asarray(row_contrib)
    n = len(d)
    se_hc = np.sqrt(np.sum(row_contrib**2) * n / max(n - k - 1, 1))
    return theta, se_hc, se_cluster



def prepare_dube_data(raw_data: pd.DataFrame) -> pd.DataFrame:
    df = raw_data[["wbcode2", "year", "y", "dem"]].copy()
    df = df.sort_values(["wbcode2", "year"])
    for k in range(1, 5):
        df[f"lag{k}y"] = shift(df, "y", k)
    lag_dem = shift(df, "dem", 1)
    df["tdemoc"] = np.nan
    df.loc[df["dem"].eq(1) & lag_dem.eq(0), "tdemoc"] = 1
    df.loc[df["dem"].eq(0) & lag_dem.eq(0), "tdemoc"] = 0
    df["Ddem"] = df["dem"] - lag_dem

    ccs0 = pd.Series(True, index=df.index)
    for k in range(1, HISTORY_LENGTH + 1):
        transition_lag = shift(df, "Ddem", k)
        # Stata's missing != 1 condition evaluates as true.
        ccs0 &= transition_lag.isna() | transition_lag.abs().ne(1)
    df["CCS_0"] = ccs0

    for h in range(1, H_POST_DUBE + 1):
        transition_lead = shift(df, "Ddem", -h)
        df[f"CCS_{h}"] = (
            df[f"CCS_{h-1}"]
            & (transition_lead.isna() | transition_lead.abs().ne(1))
        )
    df["CCS_m1"] = df["CCS_0"]
    for j in range(2, H_PRE_DUBE + 1):
        df[f"CCS_m{j}"] = (
            df[f"CCS_m{j-1}"]
            & shift(df, f"CCS_m{j-1}", 1).eq(True)
        )
    return df


def write_latex_table(
    table: pd.DataFrame,
    path: Path,
    *,
    caption: str,
    label: str,
    digits: int = 3,
) -> None:
    # Write a small dependency-free LaTeX table.
    def latex_escape(value) -> str:
        text = str(value)
        for old, new in [
            ("\\", r"\textbackslash{}"),
            ("&", r"\&"),
            ("%", r"\%"),
            ("_", r"\_"),
            ("#", r"\#"),
        ]:
            text = text.replace(old, new)
        return text

    def format_value(value) -> str:
        if pd.isna(value):
            return ""
        if isinstance(value, (float, np.floating)):
            return f"{float(value):.{digits}f}"
        return latex_escape(value)

    alignment = "l" + "r" * (len(table.columns) - 1)
    rows = [
        r"\begin{table}[!htbp]",
        r"\centering",
        rf"\caption{{{latex_escape(caption)}}}",
        rf"\label{{{latex_escape(label)}}}",
        rf"\begin{{tabular}}{{{alignment}}}",
        r"\hline",
        " & ".join(latex_escape(c) for c in table.columns)
        + r" \\",
        r"\hline",
    ]
    rows.extend(
        " & ".join(format_value(value) for value in row)
        + r" \\"
        for row in table.itertuples(index=False, name=None)
    )
    rows.extend(
        [r"\hline", r"\end{tabular}", r"\end{table}", ""]
    )
    path.write_text("\n".join(rows), encoding="utf-8")


dube_df = prepare_dube_data(raw)
BASELINE_COVARIATES = [f"lag{k}y" for k in range(1, 5)]
print(
    "Prepared panel:",
    len(dube_df),
    "rows and",
    dube_df["wbcode2"].nunique(),
    "countries.",
)


## 4. Block A — exact numerical replication of Dube et al. Figure 4


In [ ]:
if not RUN_FIGURE4_REPLICATION:
    print("RUN_FIGURE4_REPLICATION=False: skipped.")
else:
    figure4_rows = []
    figure4_specs = ("ANRR-LP", "CCC1", "CCC2", "RW-LPDID")

    for h in range(-H_PRE_DUBE, H_POST_DUBE + 1):
        if h == -1:
            for spec in figure4_specs:
                figure4_rows.append(
                    {
                        "h": h,
                        "spec": spec,
                        "estimate": 0.0,
                        "se_hc1": 0.0,
                        "se_cluster": 0.0,
                        "se_published": 0.0,
                        "rows": 0,
                        "events": 0,
                        "event_countries": 0,
                        "control_countries": 0,
                    }
                )
            continue

        if h >= 0:
            dube_df["long_y"] = (
                shift(dube_df, "y", -h) - shift(dube_df, "y", 1)
            )
        else:
            j = abs(h)
            dube_df["long_y"] = (
                shift(dube_df, "y", j) - shift(dube_df, "y", 1)
            )

        base = dube_df["tdemoc"].notna()
        if h >= 0:
            samples = {
                "ANRR-LP": base,
                "CCC1": base & dube_df["CCS_0"],
                "CCC2": (
                    (dube_df["tdemoc"].eq(1) & dube_df["CCS_0"])
                    | (
                        dube_df["tdemoc"].eq(0)
                        & dube_df[f"CCS_{h}"]
                    )
                ),
                "RW-LPDID": base & dube_df["CCS_0"],
            }
        else:
            j = abs(h)
            lead_sample = base & dube_df[f"CCS_m{j}"]
            samples = {
                "ANRR-LP": base,
                "CCC1": lead_sample,
                "CCC2": lead_sample,
                "RW-LPDID": lead_sample,
            }

        for spec, mask in samples.items():
            if spec == "RW-LPDID" and -4 <= h <= -2:
                figure4_rows.append(
                    {
                        "h": h,
                        "spec": spec,
                        "estimate": 0.0,
                        "se_hc1": 0.0,
                        "se_cluster": 0.0,
                        "se_published": 0.0,
                        "rows": 0,
                        "events": 0,
                        "event_countries": 0,
                        "control_countries": 0,
                    }
                )
                continue

            variables = [
                "long_y",
                "tdemoc",
                "wbcode2",
                "year",
                *BASELINE_COVARIATES,
            ]
            sample = (
                dube_df.loc[mask, variables]
                .dropna()
                .reset_index(drop=True)
            )
            if spec == "RW-LPDID":
                estimate, se_hc1, se_cluster = ra_atet(sample)
            else:
                estimate, se_hc1, se_cluster = ols_inference(sample)
            event = sample["tdemoc"].eq(1)
            figure4_rows.append(
                {
                    "h": h,
                    "spec": spec,
                    "estimate": estimate,
                    "se_hc1": se_hc1,
                    "se_cluster": se_cluster,
                    "se_published": (
                        se_hc1 if spec == "RW-LPDID" else se_cluster
                    ),
                    "rows": len(sample),
                    "events": int(event.sum()),
                    "event_countries": int(
                        sample.loc[event, "wbcode2"].nunique()
                    ),
                    "control_countries": int(
                        sample.loc[~event, "wbcode2"].nunique()
                    ),
                }
            )

    figure4 = pd.DataFrame(figure4_rows)
    if len(figure4) != 204:
        raise RuntimeError(
            f"Expected 204 Figure 4 rows, found {len(figure4)}."
        )

    expected_rounded = {
        ("ANRR-LP", 20): 18.856,
        ("CCC1", 20): 23.127,
        ("CCC2", 20): 17.780,
        ("RW-LPDID", 20): 23.204,
        ("ANRR-LP", 30): 19.749,
        ("CCC1", 30): 22.712,
        ("CCC2", 30): 8.639,
        ("RW-LPDID", 30): 22.607,
    }
    for (spec, h), expected in expected_rounded.items():
        value = float(
            figure4.loc[
                figure4["spec"].eq(spec) & figure4["h"].eq(h),
                "estimate",
            ].iloc[0]
        )
        if abs(value - expected) > 5e-4:
            raise RuntimeError(
                f"Figure 4 replication gate failed for {spec}, h={h}: "
                f"{value:.6f} versus {expected:.3f}."
            )

    figure4.to_csv(
        OUTPUT_DIR / "dube_figure4_diagnostic.csv", index=False
    )

    titles = {
        "ANRR-LP": "ANRR (2019) LP specification",
        "CCC1": "LP-DiD estimate (CCC1)",
        "CCC2": "LP-DiD estimate (CCC2)",
        "RW-LPDID": "Reweighted LP-DiD estimate",
    }
    fig, axes = plt.subplots(
        2, 2, figsize=(11, 7.6), sharex=True, sharey=True
    )
    for ax, spec in zip(axes.flat, titles):
        plot = figure4.loc[figure4["spec"].eq(spec)].sort_values("h")
        estimate = plot["estimate"].to_numpy(float)
        se = plot["se_published"].to_numpy(float)
        horizon = plot["h"].to_numpy(float)
        ax.fill_between(
            horizon,
            estimate - Z_975 * se,
            estimate + Z_975 * se,
            color="#6baed6",
            alpha=0.24,
            linewidth=0,
        )
        ax.plot(horizon, estimate, color="#08519c", linewidth=1.7)
        ax.axhline(0, color="black", linewidth=0.75)
        ax.set_title(titles[spec], loc="left", fontsize=10.5)
    fig.supxlabel("Years since democratization")
    fig.supylabel("GDP per capita (log × 100)")
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "dube_figure4_replication.pdf",
        bbox_inches="tight",
    )
    fig.savefig(
        OUTPUT_DIR / "dube_figure4_replication.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

    display(
        figure4.loc[
            figure4["h"].isin([-20, -10, -5, 0, 5, 10, 15, 20, 25, 30])
        ].round(3)
    )


## 5. Block B — five estimators on the Dube-compatible stack

The CCC1 eligibility rules are application-specific and therefore remain
explicit in this notebook. LPDID-RA retains the Dube et al.
clean-control regression sample, including control-only calendar
cells. The other four estimators use the common
supported calendar cells required by their propensity-score systems. All
estimates and country-cluster influence functions are computed by the
version 0.7.2 semiparametric engine. This bridge intentionally uses
internal numerical routines without expanding the package's formal public
API.

All five estimators use the same event rows. LPDID-RA follows the
literature regression-adjustment sample; the four propensity-based
estimators share the narrower supported control stack.
Pointwise intervals are country-cluster robust. Simultaneous bands use 999
country-level Rademacher multiplier draws and exclude the displayed,
non-estimated horizons \(h=-4,-3,-2,-1\).


In [ ]:
EXACT_METHODS = (
    "LPDID-RA",
    "IPW",
    "IPT",
    "DRLPDID-IPW",
    "DRLPDID-IPT",
)
EXACT_HORIZONS = tuple(range(-H_PRE_DUBE, H_POST_DUBE + 1))
BASELINE_COVARIATES = [f"lag{k}y" for k in range(1, 5)]


def dube_local_sample(df: pd.DataFrame, h: int) -> pd.DataFrame:
    if h >= 0:
        long_y = shift(df, "y", -h) - shift(df, "y", 1)
        eligible = df["tdemoc"].notna() & df["CCS_0"]
    else:
        j = abs(h)
        long_y = shift(df, "y", j) - shift(df, "y", 1)
        eligible = df["tdemoc"].notna() & df[f"CCS_m{j}"]
    columns = ["wbcode2", "year", "tdemoc", *BASELINE_COVARIATES]
    sample = df.loc[eligible, columns].copy()
    sample["long_y"] = long_y.loc[sample.index]
    return sample.dropna().reset_index(drop=True)


def common_supported_stack(sample: pd.DataFrame) -> pd.DataFrame:
    cells = sample.groupby("year")["tdemoc"].agg(["sum", "count"])
    supported_years = cells.index[
        (cells["sum"] > 0) & (cells["sum"] < cells["count"])
    ]
    local = sample.loc[sample["year"].isin(supported_years)].copy()
    local = local.rename(
        columns={"long_y": "outcome_local", "tdemoc": "D_local"}
    )
    local["D_local"] = local["D_local"].astype(float)
    if local.empty or local["D_local"].nunique() != 2:
        raise RuntimeError("The Dube-compatible local stack is one-sided.")
    return local.reset_index(drop=True)


def dube_ra_stack(sample: pd.DataFrame) -> pd.DataFrame:
    """Dube et al. RA sample: retain control-only calendar cells."""
    cells = sample.groupby("year")["tdemoc"].agg(["sum", "count"])
    treated_only = cells.index[(cells["sum"] > 0) & (cells["sum"] == cells["count"])]
    local = sample.loc[~sample["year"].isin(treated_only)].copy()
    local = local.rename(columns={"long_y": "outcome_local", "tdemoc": "D_local"})
    local["D_local"] = local["D_local"].astype(float)
    if local.empty or local["D_local"].nunique() != 2:
        raise RuntimeError("The Dube LPDID-RA stack is one-sided.")
    return local.reset_index(drop=True)


def fit_dube_v072(local: pd.DataFrame, method: str) -> dict:
    args = (local, "wbcode2", "year")
    if method == "LPDID-RA":
        return _compute_ra(*args, BASELINE_COVARIATES, True)
    if method == "IPW":
        return _compute_ipw(
            *args, BASELINE_COVARIATES, "generic", None, True, False, True
        )
    if method == "IPT":
        return _compute_ipw(
            *args, BASELINE_COVARIATES, "improved", None, True, False, True
        )
    if method == "DRLPDID-IPW":
        return _compute_dr_generic(
            *args,
            BASELINE_COVARIATES,
            BASELINE_COVARIATES,
            None,
            True,
            False,
            True,
        )
    if method == "DRLPDID-IPT":
        return _compute_dr_improved(
            *args,
            BASELINE_COVARIATES,
            BASELINE_COVARIATES,
            None,
            True,
            False,
            True,
        )
    raise ValueError(method)


def multiplier_bands(results, psi_by_method, output_dir: Path):
    rng = np.random.default_rng(MULTIPLIER_SEED)
    all_clusters = pd.Index(
        sorted(
            {
                cluster
                for by_h in psi_by_method.values()
                for influence in by_h.values()
                for cluster in influence.index
            }
        )
    )
    multipliers = rng.choice(
        [-1.0, 1.0], size=(N_MULTIPLIER, len(all_clusters))
    )
    rows = []
    for method, by_h in psi_by_method.items():
        horizons = sorted(by_h)
        psi = np.column_stack(
            [
                by_h[h].reindex(all_clusters, fill_value=0.0).to_numpy()
                for h in horizons
            ]
        )
        estimates = np.array(
            [
                results.loc[
                    results["method"].eq(method)
                    & results["horizon"].eq(h),
                    "estimate",
                ].iloc[0]
                for h in horizons
            ]
        )
        se = np.sqrt(np.sum(psi**2, axis=0))
        centered = multipliers @ psi
        critical = float(
            np.quantile(
                np.nanmax(np.abs(centered / se[None, :]), axis=1),
                1.0 - ALPHA,
            )
        )
        for h, estimate, standard_error in zip(horizons, estimates, se):
            rows.append(
                {
                    "method": method,
                    "horizon": h,
                    "critical_value": critical,
                    "sim_ci_lower": estimate - critical * standard_error,
                    "sim_ci_upper": estimate + critical * standard_error,
                    "inference_included": True,
                }
            )
    bands = pd.DataFrame(rows)
    bands.to_csv(
        output_dir / "dube_exact_dr_simultaneous_bands.csv", index=False
    )
    return bands


In [ ]:
exact_rows = []
psi_by_method = {method: {} for method in EXACT_METHODS}
dube_diagnostics = []

if not RUN_DUBE_COMPATIBLE:
    exact_event = pd.DataFrame()
    exact_scalar = pd.DataFrame()
    exact_bands = pd.DataFrame()
    exact_bands_complete = pd.DataFrame()
    print("RUN_DUBE_COMPATIBLE=False: skipped.")
else:
    for h in EXACT_HORIZONS:
        if h == BASE_PERIOD or h in DISPLAY_ONLY_LEADS:
            reason = (
                "base-period normalization"
                if h == BASE_PERIOD
                else "not separately identified with four outcome lags"
            )
            for method in EXACT_METHODS:
                exact_rows.append(
                    {
                        "horizon": h,
                        "method": method,
                        "estimate": 0.0,
                        "se": 0.0,
                        "ci_lower": 0.0,
                        "ci_upper": 0.0,
                        "eligible_rows": 0,
                        "effective_rows": 0,
                        "events": 0,
                        "event_countries": 0,
                        "control_countries": 0,
                        "estimated": False,
                        "display_reason": reason,
                    }
                )
            continue

        eligible = dube_local_sample(dube_df, h)
        local = common_supported_stack(eligible)
        ra_local = dube_ra_stack(eligible)
        n_events = int(local["D_local"].eq(1).sum())
        n_event_countries = int(
            local.loc[local["D_local"].eq(1), "wbcode2"].nunique()
        )
        n_control_countries = int(
            local.loc[local["D_local"].eq(0), "wbcode2"].nunique()
        )
        for method in EXACT_METHODS:
            method_local = ra_local if method == "LPDID-RA" else local
            n_events = int(method_local["D_local"].eq(1).sum())
            n_event_countries = int(method_local.loc[method_local["D_local"].eq(1), "wbcode2"].nunique())
            n_control_countries = int(method_local.loc[method_local["D_local"].eq(0), "wbcode2"].nunique())
            fit = fit_dube_v072(method_local, method)
            estimate = float(fit["estimate"])
            standard_error = float(fit["se"])
            exact_rows.append(
                {
                    "horizon": h,
                    "method": method,
                    "estimate": estimate,
                    "se": standard_error,
                    "ci_lower": estimate - Z_975 * standard_error,
                    "ci_upper": estimate + Z_975 * standard_error,
                    "eligible_rows": len(eligible),
                    "effective_rows": len(method_local),
                    "events": n_events,
                    "event_countries": n_event_countries,
                    "control_countries": n_control_countries,
                    "estimated": True,
                    "display_reason": "",
                }
            )
            psi_by_method[method][h] = fit["psi_by_cluster"]
            if method in {"IPT", "DRLPDID-IPT"}:
                dube_diagnostics.append(
                    {
                        "horizon": h,
                        "method": method,
                        **{
                            key: value
                            for key, value in fit.items()
                            if key.startswith("ipt_")
                            and np.isscalar(value)
                        },
                    }
                )

    exact_event = pd.DataFrame(exact_rows)
    exact_event.to_csv(
        OUTPUT_DIR / "dube_exact_dr_event_study.csv", index=False
    )
    pd.DataFrame(dube_diagnostics).to_csv(
        OUTPUT_DIR / "dube_exact_dr_nuisance_diagnostics.csv", index=False
    )

    event_counts = (
        exact_event.loc[exact_event["estimated"]]
        .pivot(index="horizon", columns="method", values="events")
    )
    if not event_counts.eq(event_counts.iloc[:, 0], axis=0).all().all():
        raise RuntimeError("The five methods do not use the same event rows.")

    estimates = exact_event.pivot(
        index="horizon", columns="method", values="estimate"
    )
    nested = pd.DataFrame(
        {
            "horizon": estimates.index,
            "ipt_estimate": estimates["IPT"].to_numpy(),
            "dript_estimate": estimates["DRLPDID-IPT"].to_numpy(),
        }
    )
    nested["absolute_difference"] = (
        nested["ipt_estimate"] - nested["dript_estimate"]
    ).abs()
    tolerance_by_h = {
        int(row["horizon"]): float(row["ipt_dript_identity_tolerance"])
        for row in dube_diagnostics
        if row["method"] == "DRLPDID-IPT"
    }
    nested["certified_tolerance"] = nested["horizon"].map(
        lambda h: 0.0 if h in DISPLAY_ONLY_LEADS | {BASE_PERIOD}
        else tolerance_by_h[int(h)]
    )
    nested["passes_certified_identity"] = (
        nested["absolute_difference"] <= nested["certified_tolerance"]
    )
    nested.to_csv(
        OUTPUT_DIR / "dube_exact_nested_basis_identity.csv", index=False
    )
    if not nested["passes_certified_identity"].all():
        failed = nested.loc[~nested["passes_certified_identity"]]
        raise RuntimeError(
            "The 0.7.2 IPT–DRLPDID-IPT identity failed at horizons "
            f"{failed['horizon'].astype(int).tolist()}."
        )

    cluster_universe = pd.Index(
        sorted(
            {
                cluster
                for method in psi_by_method.values()
                for influence in method.values()
                for cluster in influence.index
            }
        )
    )
    scalar_rows = []
    for method in EXACT_METHODS:
        post = exact_event.loc[
            exact_event["method"].eq(method)
            & exact_event["horizon"].between(*POLICY_WINDOW)
        ]
        estimate = float(post["estimate"].mean())
        weights = 1.0 / (POLICY_WINDOW[1] - POLICY_WINDOW[0] + 1)
        scalar_psi = sum(
            (
                psi_by_method[method][h].reindex(
                    cluster_universe, fill_value=0.0
                )
                for h in range(POLICY_WINDOW[0], POLICY_WINDOW[1] + 1)
            ),
            start=pd.Series(0.0, index=cluster_universe),
        ) * weights
        standard_error = float(
            np.sqrt(np.sum(scalar_psi.to_numpy() ** 2))
        )
        scalar_rows.append(
            {
                "estimator": method,
                "estimate": estimate,
                "se": standard_error,
                "ci_lower": estimate - Z_975 * standard_error,
                "ci_upper": estimate + Z_975 * standard_error,
                "window_start": POLICY_WINDOW[0],
                "window_end": POLICY_WINDOW[1],
            }
        )
    exact_scalar = pd.DataFrame(scalar_rows)
    exact_scalar.to_csv(
        OUTPUT_DIR / "table_democracy_scalar.csv", index=False
    )
    write_latex_table(
        exact_scalar,
        OUTPUT_DIR / "table_democracy_scalar.tex",
        caption="Democratization: Dube-compatible average onset effects",
        label="tab:democracy-scalar",
    )

    old_v064 = pd.DataFrame(
        [
            ("LPDID-RA", 1.996950509918446, 2.6167814704478385),
            ("IPW", 1.8673401025463912, 2.886056370821113),
            ("IPT", 1.2418357426235795, 2.7902315916993548),
            ("DRLPDID-IPW", 0.5390946555330299, 3.216320402723943),
            ("DRLPDID-IPT", 1.241835742631364, 2.8210353667911767),
        ],
        columns=["estimator", "v064_estimate", "v064_se"],
    )
    documentary = exact_scalar.merge(
        old_v064, on="estimator", validate="one_to_one"
    )
    documentary["estimate_change"] = (
        documentary["estimate"] - documentary["v064_estimate"]
    )
    documentary["se_change"] = documentary["se"] - documentary["v064_se"]
    documentary.to_csv(
        OUTPUT_DIR / "dube_v072_vs_v064_documentary.csv", index=False
    )

    exact_bands = multiplier_bands(
        exact_event, psi_by_method, OUTPUT_DIR
    )
    display_rows = []
    for method in EXACT_METHODS:
        critical = float(
            exact_bands.loc[
                exact_bands["method"].eq(method), "critical_value"
            ].iloc[0]
        )
        for h in sorted(DISPLAY_ONLY_LEADS | {BASE_PERIOD}):
            display_rows.append(
                {
                    "method": method,
                    "horizon": h,
                    "critical_value": critical,
                    "sim_ci_lower": 0.0,
                    "sim_ci_upper": 0.0,
                    "inference_included": False,
                }
            )
    exact_bands_complete = (
        pd.concat(
            [exact_bands, pd.DataFrame(display_rows)], ignore_index=True
        )
        .sort_values(["method", "horizon"])
        .reset_index(drop=True)
    )
    exact_bands_complete.to_csv(
        OUTPUT_DIR / "dube_exact_dr_simultaneous_bands_complete.csv",
        index=False,
    )

    print("Dube-compatible 0.7.2 scalar estimates")
    display(exact_scalar.round(4))
    print("Documentary comparison with the former 0.6.4 run")
    display(documentary.round(6))


## 6. Dube-compatible pointwise intervals and simultaneous bands


In [ ]:
if RUN_DUBE_COMPATIBLE:
    colors = {
        "LPDID-RA": ("#d95f0e", "#fdae6b"),
        "DRLPDID-IPT": ("#08519c", "#6baed6"),
    }

    # Panel (a): pointwise country-cluster-robust intervals.
    fig, ax = plt.subplots(figsize=(9.2, 5.8))
    for method in ("LPDID-RA", "DRLPDID-IPT"):
        path = exact_event.loc[
            exact_event["method"].eq(method)
        ].sort_values("horizon")
        line_color, fill_color = colors[method]
        ax.fill_between(
            path["horizon"],
            path["ci_lower"],
            path["ci_upper"],
            color=fill_color,
            alpha=0.25,
            linewidth=0,
        )
        ax.plot(
            path["horizon"],
            path["estimate"],
            color=line_color,
            linewidth=1.8,
            label=method,
        )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(
        xlabel="Years since democratization",
        ylabel="GDP per capita (log × 100)",
        title=(
            "Dube-compatible democracy-onset effects\n"
            "Pointwise 95% country-cluster-robust confidence intervals"
        ),
        xlim=(-H_PRE_DUBE, H_POST_DUBE),
    )
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "figure1a_dube_RA_DRIPT_cluster_ci.pdf",
        bbox_inches="tight",
    )
    fig.savefig(
        OUTPUT_DIR / "figure1a_dube_RA_DRIPT_cluster_ci.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

    # Panel (b): simultaneous country-level multiplier bands.
    fig, ax = plt.subplots(figsize=(9.2, 5.8))
    for method in ("LPDID-RA", "DRLPDID-IPT"):
        path = exact_event.loc[
            exact_event["method"].eq(method),
            ["horizon", "estimate"],
        ].merge(
            exact_bands_complete.loc[
                exact_bands_complete["method"].eq(method),
                [
                    "horizon",
                    "sim_ci_lower",
                    "sim_ci_upper",
                ],
            ],
            on="horizon",
            how="left",
            validate="one_to_one",
        ).sort_values("horizon")
        if path[["sim_ci_lower", "sim_ci_upper"]].isna().any().any():
            raise RuntimeError(
                f"Incomplete simultaneous band for {method}."
            )
        line_color, fill_color = colors[method]
        ax.fill_between(
            path["horizon"],
            path["sim_ci_lower"],
            path["sim_ci_upper"],
            color=fill_color,
            alpha=0.25,
            linewidth=0,
        )
        ax.plot(
            path["horizon"],
            path["estimate"],
            color=line_color,
            linewidth=1.8,
            label=method,
        )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(
        xlabel="Years since democratization",
        ylabel="GDP per capita (log × 100)",
        title=(
            "Dube-compatible democracy-onset effects\n"
            "Simultaneous 95% country-level Rademacher multiplier bands"
        ),
        xlim=(-H_PRE_DUBE, H_POST_DUBE),
    )
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "figure1b_dube_RA_DRIPT_multiplier_band.pdf",
        bbox_inches="tight",
    )
    fig.savefig(
        OUTPUT_DIR / "figure1b_dube_RA_DRIPT_multiplier_band.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

else:
    print("Dube-compatible figures skipped.")


## 7. Dube-compatible all-method paths and support diagnostics


In [ ]:
if RUN_DUBE_COMPATIBLE:
    styles = {
        "LPDID-RA": ("black", "-", "o"),
        "IPW": ("#D55E00", "--", None),
        "IPT": ("#0072B2", "--", None),
        "DRLPDID-IPW": ("#CC79A7", "-.", None),
        "DRLPDID-IPT": ("#009E73", "-", "s"),
    }
    fig, ax = plt.subplots(figsize=(9.2, 5.8))
    for method in EXACT_METHODS:
        path = exact_event.loc[
            exact_event["method"].eq(method)
        ].sort_values("horizon")
        color, linestyle, marker = styles[method]
        ax.plot(
            path["horizon"],
            path["estimate"],
            color=color,
            linestyle=linestyle,
            marker=marker,
            markevery=5,
            markersize=3.5,
            linewidth=1.6,
            label=method,
        )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(
        xlabel="Years since democratization",
        ylabel="GDP per capita (log × 100)",
        title="Dube-compatible onset effects: all five estimators",
        xlim=(-H_PRE_DUBE, H_POST_DUBE),
    )
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "dube_exact_dr_all_methods.pdf",
        bbox_inches="tight",
    )
    fig.savefig(
        OUTPUT_DIR / "dube_exact_dr_all_methods.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

    article_horizons = exact_event.loc[
        exact_event["method"].isin(["LPDID-RA", "DRLPDID-IPT"])
        & exact_event["horizon"].isin([10, 20, 30]),
        ["horizon", "method", "estimate", "se", "ci_lower", "ci_upper"],
    ].sort_values(["horizon", "method"])
    article_horizons.to_csv(
        OUTPUT_DIR / "article_horizon_crosscheck.csv", index=False
    )
    display(article_horizons.round(3))

    support = (
        exact_event.loc[
            exact_event["method"].eq("LPDID-RA"),
            [
                "horizon",
                "eligible_rows",
                "effective_rows",
                "events",
                "event_countries",
                "control_countries",
                "estimated",
            ],
        ]
        .sort_values("horizon")
        .reset_index(drop=True)
    )
    support.to_csv(
        OUTPUT_DIR / "dube_compatible_support_by_horizon.csv",
        index=False,
    )
    display(
        support.loc[support["horizon"].isin([0, 5, 10, 15, 20, 25, 30])]
    )

else:
    print("Dube-compatible diagnostics skipped.")


## 8. Block C — formal sustained-switch-in specification

This block uses only the public 0.7.2 API:

- `design="switching"`;
- observed clean histories of length \(L\);
- treatment sustained through every post-entry horizon;
- stable-zero and stabilized stable-one all-stayers;
- country-cluster-robust pointwise inference;
- simultaneous country-level Rademacher multiplier bands.

The main specification uses \(L=20\). The \(L=9\) run is retained as an
optional replication sensitivity and does not change the formal API.


In [ ]:
FORMAL_METHODS = {
    "DRLPDID-RA": "ra",
    "IPW": "ipw",
    "IPT": "ipt",
    "DRLPDID-IPW": "dr-ipw",
    "DRLPDID-IPT": "dr-ipt",
}


def fit_formal(
    label: str,
    method: str,
    history_length: int,
    inference: str = "cluster",
):
    start = time.perf_counter()
    result = DRLPDID(
        estimation_method=method,
        design="switching",
        stabilization_window=history_length,
        horizons=(-FORMAL_H_PRE, FORMAL_H_POST),
        post_window=POLICY_WINDOW,
        inference=inference,
        n_bootstrap=N_MULTIPLIER,
        alpha=ALPHA,
        seed=MULTIPLIER_SEED,
    ).fit(
        raw,
        outcome="y",
        unit="wbcode2",
        time="year",
        treatment="dem",
        covariates=BASELINE_COVARIATES,
    )
    return result, time.perf_counter() - start


def policy_scalar(result, label: str, history_length: int, seconds: float):
    term = f"ATT policy-window [{POLICY_WINDOW[0]},{POLICY_WINDOW[1]}]"
    selected = result.scalars.loc[result.scalars["term"].eq(term)]
    if len(selected) != 1:
        raise RuntimeError(f"{label}, L={history_length}: scalar is absent.")
    row = selected.iloc[0]
    return {
        "L": history_length,
        "estimator": label,
        "estimate": float(row["estimate"]),
        "se": float(row["se"]),
        "p_value": float(row["p_value"]),
        "ci_lower": float(row["ci_lower"]),
        "ci_upper": float(row["ci_upper"]),
        "fit_seconds": float(seconds),
    }


In [ ]:
formal_event = pd.DataFrame()
formal_scalar = pd.DataFrame()
formal_composition = pd.DataFrame()
formal_joint_paths = {}
formal_results = {}
formal_audit = {
    "requested": RUN_FORMAL_SWITCHING,
    "completed": False,
    "error": None,
}

if not RUN_FORMAL_SWITCHING:
    print("RUN_FORMAL_SWITCHING=False: skipped.")
else:
    history_lengths = [FORMAL_L]
    if RUN_FORMAL_L9:
        history_lengths.append(FORMAL_L9)

    event_frames = []
    scalar_rows = []
    for history_length in history_lengths:
        for label, method in FORMAL_METHODS.items():
            print(f"Formal fit: L={history_length}, {label}")
            result, seconds = fit_formal(
                label, method, history_length, inference="cluster"
            )
            formal_results[(history_length, label)] = result
            event = result.event_study.copy()
            event["L"] = history_length
            event["estimator"] = label
            event["fit_seconds"] = seconds
            event_frames.append(event)
            scalar_rows.append(
                policy_scalar(result, label, history_length, seconds)
            )

    formal_event = pd.concat(event_frames, ignore_index=True)
    formal_scalar = pd.DataFrame(scalar_rows)

    count_columns = [
        "horizon",
        "n_event_rows",
        "n_event_units",
        "n_control_rows",
        "n_control_units",
        "n_reference_dates",
        "n_stable_zero_control_rows",
        "n_stable_one_control_rows",
    ]
    composition_frames = []
    for history_length in history_lengths:
        reference = formal_results[
            (history_length, "DRLPDID-RA")
        ].event_study[count_columns].reset_index(drop=True)
        for label in list(FORMAL_METHODS)[1:]:
            other = formal_results[
                (history_length, label)
            ].event_study[count_columns].reset_index(drop=True)
            pd.testing.assert_frame_equal(
                reference, other, check_dtype=False
            )
        composition_frames.append(
            reference.rename(
                columns={"n_reference_dates": "calendar_cells"}
            ).assign(L=history_length)
        )
    formal_composition = pd.concat(
        composition_frames, ignore_index=True
    )

    identity_rows = []
    for history_length in history_lengths:
        ipt = formal_results[
            (history_length, "IPT")
        ].event_study[["horizon", "estimate"]]
        dript_result = formal_results[
            (history_length, "DRLPDID-IPT")
        ]
        dript = dript_result.event_study[["horizon", "estimate"]]
        identity = ipt.merge(
            dript,
            on="horizon",
            suffixes=("_ipt", "_dript"),
            validate="one_to_one",
        )
        identity["L"] = history_length
        identity["absolute_difference"] = (
            identity["estimate_ipt"] - identity["estimate_dript"]
        ).abs()
        diagnostics = dript_result.metadata["nuisance_diagnostics"]
        identity["certified_tolerance"] = identity["horizon"].map(
            lambda h: 0.0
            if int(h) == BASE_PERIOD
            else diagnostics[int(h)]["ipt_dript_identity_tolerance"]
        )
        identity["passes_certified_identity"] = (
            identity["absolute_difference"]
            <= identity["certified_tolerance"]
        )
        identity_rows.append(identity)
    formal_identity = pd.concat(identity_rows, ignore_index=True)
    if not formal_identity["passes_certified_identity"].all():
        failed = formal_identity.loc[
            ~formal_identity["passes_certified_identity"]
        ]
        raise RuntimeError(
            "Formal IPT–DRLPDID-IPT identity failed at "
            f"{failed[['L', 'horizon']].to_dict('records')}."
        )

    formal_event.to_csv(
        OUTPUT_DIR / "formal_switching_event_study.csv", index=False
    )
    formal_event.to_csv(
        OUTPUT_DIR / "formal_event_study_cluster.csv", index=False
    )
    formal_scalar.to_csv(
        OUTPUT_DIR / "table_formal_switching_scalar.csv", index=False
    )
    formal_scalar.to_csv(
        OUTPUT_DIR / "formal_scalar_policy_window.csv", index=False
    )
    formal_composition.to_csv(
        OUTPUT_DIR / "formal_switching_composition.csv", index=False
    )
    formal_composition.to_csv(
        OUTPUT_DIR / "formal_horizon_composition.csv", index=False
    )
    formal_identity.to_csv(
        OUTPUT_DIR / "formal_nested_basis_identity.csv", index=False
    )

    main_table = formal_scalar.loc[
        formal_scalar["L"].eq(FORMAL_L)
    ].copy()
    write_latex_table(
        main_table,
        OUTPUT_DIR / "table_democracy_formal_scalar.tex",
        caption="Democratization: formal sustained-switch-in effects",
        label="tab:democracy-formal-scalar",
    )
    if RUN_FORMAL_L9:
        l9_table = formal_scalar.loc[
            formal_scalar["L"].eq(FORMAL_L9)
        ].copy()
        write_latex_table(
            l9_table,
            OUTPUT_DIR / "table_democracy_formal_L9.tex",
            caption="Formal design: sensitivity to clean-history length",
            label="tab:democracy-formal-L9",
        )

    old_formal = pd.DataFrame(
        [
            (20, "DRLPDID-RA", 1.384, 3.908),
            (20, "IPW", 4.289, 8.036),
            (20, "IPT", 0.077, 4.283),
            (20, "DRLPDID-IPW", -2.222, 6.344),
            (20, "DRLPDID-IPT", 0.080, 4.342),
            (9, "DRLPDID-RA", 2.762, 2.927),
            (9, "IPW", 5.301, 5.640),
            (9, "IPT", 1.406, 3.266),
            (9, "DRLPDID-IPW", 0.358, 3.930),
            (9, "DRLPDID-IPT", 1.403, 3.297),
        ],
        columns=["L", "estimator", "v064_estimate", "v064_se"],
    )
    formal_documentary = formal_scalar.merge(
        old_formal,
        on=["L", "estimator"],
        how="left",
        validate="one_to_one",
    )
    formal_documentary["estimate_change"] = (
        formal_documentary["estimate"]
        - formal_documentary["v064_estimate"]
    )
    formal_documentary["se_change"] = (
        formal_documentary["se"] - formal_documentary["v064_se"]
    )
    formal_documentary.to_csv(
        OUTPUT_DIR / "formal_v072_vs_v064_documentary.csv", index=False
    )
    formal_documentary.to_csv(
        OUTPUT_DIR / "formal_article_number_comparison.csv", index=False
    )

    for label, method in [
        ("DRLPDID-RA", "ra"),
        ("DRLPDID-IPT", "dr-ipt"),
    ]:
        multiplier, seconds = fit_formal(
            label, method, FORMAL_L, inference="multiplier"
        )
        point = formal_results[
            (FORMAL_L, label)
        ].event_study[
            ["horizon", "estimate", "se", "ci_lower", "ci_upper"]
        ]
        band = multiplier.event_study[
            ["horizon", "estimate", "sim_ci_lower", "sim_ci_upper"]
        ].rename(columns={"estimate": "multiplier_estimate"})
        joint = point.merge(
            band, on="horizon", validate="one_to_one"
        )
        if not np.allclose(
            joint["estimate"],
            joint["multiplier_estimate"],
            atol=1e-10,
            equal_nan=True,
        ):
            raise RuntimeError(
                f"{label}: point estimates changed across inference modes."
            )
        joint = joint.drop(columns="multiplier_estimate")
        joint.loc[
            joint["horizon"].eq(BASE_PERIOD),
            ["sim_ci_lower", "sim_ci_upper"],
        ] = 0.0
        formal_joint_paths[label] = joint
        joint.to_csv(
            OUTPUT_DIR
            / f"{label.lower().replace('-', '_')}_formal_joint_inference.csv",
            index=False,
        )
        if label == "DRLPDID-IPT":
            multiplier.event_study.to_csv(
                OUTPUT_DIR / "formal_dript_multiplier_path.csv",
                index=False,
            )

    formal_audit.update(
        {
            "completed": True,
            "main_L": FORMAL_L,
            "sensitivity_L": FORMAL_L9 if RUN_FORMAL_L9 else None,
            "horizons": [-FORMAL_H_PRE, FORMAL_H_POST],
            "policy_window": list(POLICY_WINDOW),
            "design": "formal sustained switch-in",
            "control_pool": "stabilized all-stayers",
            "package_version": pydrlpdid.__version__,
            "multiplier_replications": N_MULTIPLIER,
        }
    )
    display(main_table.round(4))
    if RUN_FORMAL_L9:
        display(l9_table.round(4))
    display(
        formal_composition.loc[
            formal_composition["L"].eq(FORMAL_L)
            & formal_composition["horizon"].isin([0, 5, 10, 15])
        ]
    )

(OUTPUT_DIR / "formal_application_audit.json").write_text(
    json.dumps(formal_audit, indent=2), encoding="utf-8"
)


## 9. Formal-design figures


In [ ]:
if RUN_FORMAL_SWITCHING and formal_audit["completed"]:
    colors = {"DRLPDID-RA": "#d95f0e", "DRLPDID-IPT": "#08519c"}

    fig, axes = plt.subplots(
        1, 2, figsize=(12, 4.8), sharex=True, sharey=True
    )
    for axis, interval in zip(axes, ["pointwise", "simultaneous"]):
        for label, path in formal_joint_paths.items():
            lower = "ci_lower" if interval == "pointwise" else "sim_ci_lower"
            upper = "ci_upper" if interval == "pointwise" else "sim_ci_upper"
            axis.fill_between(
                path["horizon"],
                path[lower],
                path[upper],
                color=colors[label],
                alpha=0.16,
            )
            axis.plot(
                path["horizon"],
                path["estimate"],
                color=colors[label],
                linewidth=1.6,
                label=label,
            )
        axis.axhline(0, color="black", linewidth=0.8)
        axis.axvline(-0.5, color="black", linestyle="--", linewidth=0.8)
        axis.set_title(
            "Pointwise 95% country-cluster-robust CIs"
            if interval == "pointwise"
            else "Simultaneous 95% multiplier bands"
        )
        axis.set_xlabel("Event-study horizon")
    axes[0].set_ylabel("Effect on log GDP per capita × 100")
    axes[0].legend(frameon=False)
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "figure_formal_switching_RA_DRIPT.pdf",
        bbox_inches="tight",
    )
    fig.savefig(
        OUTPUT_DIR / "figure_formal_switching_RA_DRIPT.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

    formal_l20 = formal_event.loc[
        formal_event["L"].eq(FORMAL_L)
    ].copy()
    post = formal_l20.loc[
        formal_l20["horizon"].between(0, FORMAL_H_POST)
    ]
    dript_band = formal_joint_paths["DRLPDID-IPT"].loc[
        formal_joint_paths["DRLPDID-IPT"]["horizon"].between(
            0, FORMAL_H_POST
        )
    ]

    fig, ax = plt.subplots(figsize=(8.8, 5.4))
    for estimator_name, color in [
        ("DRLPDID-RA", "#d95f0e"),
        ("IPT", "#756bb1"),
        ("DRLPDID-IPT", "#08519c"),
    ]:
        path = post.loc[
            post["estimator"].eq(estimator_name)
        ].sort_values("horizon")
        ax.plot(
            path["horizon"],
            path["estimate"],
            color=color,
            marker="o",
            markersize=3,
            linewidth=1.6,
            label=estimator_name,
        )
    ax.fill_between(
        dript_band["horizon"],
        dript_band["sim_ci_lower"],
        dript_band["sim_ci_upper"],
        color="#6baed6",
        alpha=0.22,
        linewidth=0,
        label="DRLPDID-IPT simultaneous 95% band",
    )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(
        xlabel="Years since democratization",
        ylabel="GDP per capita (log × 100)",
        title=f"Formal sustained-democratization path (L={FORMAL_L})",
    )
    ax.legend(frameon=False, fontsize=8.5)
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "dube_democracy_main_path.pdf", bbox_inches="tight"
    )
    fig.savefig(
        OUTPUT_DIR / "dube_democracy_main_path.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

    fig, ax = plt.subplots(figsize=(8.8, 5.4))
    for estimator_name in FORMAL_METHODS:
        path = post.loc[
            post["estimator"].eq(estimator_name)
        ].sort_values("horizon")
        ax.plot(
            path["horizon"],
            path["estimate"],
            marker="o",
            markersize=2.5,
            linewidth=1.4,
            label=estimator_name,
        )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(
        xlabel="Years since democratization",
        ylabel="GDP per capita (log × 100)",
        title=f"Formal sustained-switch-in paths (L={FORMAL_L})",
    )
    ax.legend(frameon=False, fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "dube_democracy_all_methods.pdf", bbox_inches="tight"
    )
    fig.savefig(
        OUTPUT_DIR / "dube_democracy_all_methods.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()
else:
    print("Formal figures skipped.")


## 10. Reproducibility manifest and output archive


In [ ]:
dependency_versions = {}
for package in [
    "numpy",
    "pandas",
    "scipy",
    "statsmodels",
    "patsy",
    "matplotlib",
]:
    try:
        dependency_versions[package] = importlib_metadata.version(package)
    except importlib_metadata.PackageNotFoundError:
        dependency_versions[package] = None

manifest = {
    "project_dir": str(PROJECT_DIR),
    "python": sys.version,
    "platform": platform.platform(),
    "pydrlpdid_version": pydrlpdid.__version__,
    "pydrlpdid_source": str(PACKAGE_FILE),
    "pydrlpdid_source_hashes": PACKAGE_SOURCE_HASHES,
    "dependency_versions": dependency_versions,
    "data": data_audit,
    "published_figure4_replication": {
        "run": RUN_FIGURE4_REPLICATION,
        "role": "literature benchmark; not a DRLPDID API call",
    },
    "dube_compatible_design": {
        "run": RUN_DUBE_COMPATIBLE,
        "estimand": "clean-past democracy onset under realized future paths",
        "history_length": HISTORY_LENGTH,
        "history_boundary": "available-history",
        "future_treatment_conditioning": False,
        "common_event_rows": True,
        "lpdid_ra_sample": "Dube clean-control regression sample",
        "propensity_estimators_common_supported_calendar_cells": True,
        "numerical_engine": "pydrlpdid 0.7.2 internal moment routines",
        "horizons": [-H_PRE_DUBE, H_POST_DUBE],
        "base_period": BASE_PERIOD,
        "display_only_unidentified_leads": sorted(DISPLAY_ONLY_LEADS),
        "policy_window": list(POLICY_WINDOW),
        "cluster": "country",
        "multiplier_replications": N_MULTIPLIER,
        "multiplier_seed": MULTIPLIER_SEED,
    },
    "formal_switching_design": formal_audit,
}
manifest_path = OUTPUT_DIR / "run_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

archive_path = Path(
    shutil.make_archive(
        str(PROJECT_DIR / OUTPUT_FOLDER),
        "zip",
        root_dir=PROJECT_DIR,
        base_dir=OUTPUT_FOLDER,
    )
)
manifest["outputs"] = sorted(
    path.name for path in OUTPUT_DIR.iterdir() if path.is_file()
)
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Run completed.")
print("Generated files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(" -", path.name)
print("Output archive:", archive_path)


## Guide to outputs

### Published replication

- `dube_figure4_diagnostic.csv`
- `dube_figure4_replication.pdf/.png`

### Dube-compatible 0.7.2 extension

- `table_democracy_scalar.csv/.tex`
- `dube_exact_dr_event_study.csv`
- `dube_exact_dr_nuisance_diagnostics.csv`
- `dube_exact_nested_basis_identity.csv`
- `dube_exact_dr_simultaneous_bands.csv`
- `dube_exact_dr_simultaneous_bands_complete.csv`
- `figure1a_dube_RA_DRIPT_cluster_ci.pdf/.png`
- `figure1b_dube_RA_DRIPT_multiplier_band.pdf/.png`
- `dube_exact_dr_all_methods.pdf/.png`
- `article_horizon_crosscheck.csv`
- `dube_compatible_support_by_horizon.csv`
- `dube_v072_vs_v064_documentary.csv`

### Formal switching application

- `formal_switching_event_study.csv`
- `formal_event_study_cluster.csv`
- `table_formal_switching_scalar.csv`
- `formal_scalar_policy_window.csv`
- `table_democracy_formal_scalar.tex`
- `table_democracy_formal_L9.tex` when the \(L=9\) sensitivity runs
- `formal_switching_composition.csv`
- `formal_horizon_composition.csv`
- `formal_nested_basis_identity.csv`
- `drlpdid_ra_formal_joint_inference.csv`
- `drlpdid_ipt_formal_joint_inference.csv`
- `formal_dript_multiplier_path.csv`
- `figure_formal_switching_RA_DRIPT.pdf/.png`
- `dube_democracy_main_path.pdf/.png`
- `dube_democracy_all_methods.pdf/.png`
- `formal_v072_vs_v064_documentary.csv`
- `formal_article_number_comparison.csv` (compatibility alias for the
  same non-blocking documentary comparison)
- `formal_application_audit.json`

`run_manifest.json` records the data hash, package source hashes,
dependency versions, design distinctions, and inference settings.
The complete output folder is also archived as
`application_2_dube_v072.zip`.
